In [1]:
!pip install ultralytics tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.3 MB/s eta 0:00:00


In [2]:
import os
import re
import shutil
import glob
import numpy as np
from tqdm.auto import tqdm
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from scipy.signal import savgol_filter
import torch

class GaitPreprocessor:
    def __init__(self, target_frames=60): # 60 frame
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = YOLO('yolov8n-pose.pt').to(self.device)
        self.target_frames = target_frames
        
        
        self.indices = list(range(17)) 
        
        self.data_store = {}  # {angle: {sid: [{video, seq}] }}

    def process_video(self, video_path):
        
        results = self.model(video_path, stream=True, verbose=False, device=self.device, conf=0.3)
        raw_seq = []
        
        for res in results:
            # Controllo base esistenza
            if res.keypoints is None: continue
            
            #FIX CRASH
            if not hasattr(res.keypoints, 'xyn') or res.keypoints.xyn.shape[0] == 0:
                continue
            # -------------------------------------------------

            #Estrae (17, 2) Coordinate
            kps = res.keypoints.xyn[0].cpu().numpy()
            
            # Estrae (17, 1) Confidenza
            if res.keypoints.conf is not None:
                confs = res.keypoints.conf[0].cpu().numpy()[:, None]
            else:
                confs = np.ones((17, 1)) # Fallback se manca la confidenza
            
            # Uniamo tutto in (17, 3) -> [x, y, conf]
            kp_complete = np.hstack([kps, confs])

            # Controllo qualità: se la media conf è < 0.4 scarta il frame
            if np.mean(kp_complete[:, 2]) < 0.4: continue
            
            raw_seq.append(kp_complete)
            # Limite massimo frame per evitare file giganti
            if len(raw_seq) >= 300: break
            
        if len(raw_seq) < 30: return None
        
        data = np.array(raw_seq) # (T, 17, 3)

        #NORMALIZZAZIONE ALTEZZA
        #Centriamo sui fianchi (11, 12)
        hips_center_x = (data[:, 11, 0] + data[:, 12, 0]) / 2.0
        hips_center_y = (data[:, 11, 1] + data[:, 12, 1]) / 2.0
        
        y_shoulders = (data[:, 5, 1] + data[:, 6, 1]) / 2.0
        y_ankles = (data[:, 15, 1] + data[:, 16, 1]) / 2.0
        heights = np.abs(y_ankles - y_shoulders) + 1e-6
        
        #Centra e scala
        data[:, :, 0] = (data[:, :, 0] - hips_center_x[:, None]) / heights[:, None]
        data[:, :, 1] = (data[:, :, 1] - hips_center_y[:, None]) / heights[:, None]

        #SMOOTHING
        window_len = 7
        if len(data) < window_len: window_len = len(data) // 2 * 2 + 1
        
        if window_len > 3:
            for k in range(17):
                for c in range(2):
                    try:
                        data[:, k, c] = savgol_filter(data[:, k, c], window_length=window_len, polyorder=2)
                    except: pass

        #CAMPIONAMENTO
        if len(data) != self.target_frames:
            idx = np.linspace(0, len(data)-1, self.target_frames).astype(int)
            data = data[idx]
            
        return data.astype(np.float32)


    def extract_angle_id(self, filename, ds_name):
        f = filename.lower()
        angle = None
        sid = None
        
        #LOGICA PER ESTRAZIONE ID
        if ds_name=="LargeGait":
            
            match_angle = re.search(r'_(0|45|90)', f)
            angle = match_angle.group(1) if match_angle else "90"
            
            match_id = re.search(r'p(\d+)', f)
            #SID univoco: LargeGait_p12
            sid = f"{ds_name}_p{match_id.group(1).lstrip('0')}" if match_id else None
            
        elif ds_name=="RealWorld":
            match_angle = re.search(r'_(0|45|90)_', f)
            angle = match_angle.group(1) if match_angle else "90"
            match_id = re.search(r'p(\d+)', f)
            sid = f"{ds_name}_p{match_id.group(1).lstrip('0')}" if match_id else None
            
        elif ds_name=="SMVDU":
            match_angle = re.match(r'\d+_(\d)_', f)
            if match_angle:
                a = match_angle.group(1)
                angle = "90" if a in ["1","2","3","4"] else "0"
            match_id = re.match(r'(\d+)_', f)
            sid = f"{ds_name}_p{match_id.group(1).lstrip('0')}" if match_id else None
            
        return angle, sid

    def run(self, datasets, out_base="/kaggle/working/processed_gait_multiangle"):
        if os.path.exists(out_base): shutil.rmtree(out_base)
        all_videos = []

        print(" Scansione Dataset...")
        for ds_name, ds_path in datasets.items():
            for root, _, files in os.walk(ds_path):
                for f in files:
                    if not f.lower().endswith(('.mp4','.avi','.mov')): continue
                    angle, sid = self.extract_angle_id(f, ds_name)
                    if angle is None or sid is None: continue
                    all_videos.append((os.path.join(root,f), ds_name, angle, sid))

        #Statistiche
        angle_summary = {}
        for _, _, angle, sid in all_videos:
            if angle not in angle_summary: angle_summary[angle] = {"videos":0, "ids":set()}
            angle_summary[angle]["videos"] += 1
            angle_summary[angle]["ids"].add(sid)

        print(" Riepilogo iniziale:")
        for angle, v in angle_summary.items():
            print(f" - Angle {angle}: {v['videos']} video, {len(v['ids'])} identità")

        #Processamento
        print(" Elaborazione Video...")
        for v_path, ds_name, angle, sid in tqdm(all_videos):
            seq = self.process_video(v_path)
            if seq is None: continue
            
            if angle not in self.data_store: self.data_store[angle] = {}
            if sid not in self.data_store[angle]: self.data_store[angle][sid] = []
            self.data_store[angle][sid].append(seq)

        # Split Train/Val/Test
        # Uniamo tutti i SID trovati in qualsiasi angolo
        valid_sids = set()
        for angle_dict in self.data_store.values():
            valid_sids.update(angle_dict.keys())
        
        sids_list = sorted(list(valid_sids))
        print(f"\n Totale identità valide uniche: {len(sids_list)}")

        train_ids, tmp_ids = train_test_split(sids_list, test_size=0.3, random_state=42)
        val_ids, test_ids = train_test_split(tmp_ids, test_size=0.5, random_state=42)
        
        #Mappa veloce ID -> Split
        id_to_split = {sid: "train" for sid in train_ids}
        id_to_split.update({sid: "val" for sid in val_ids})
        id_to_split.update({sid: "test" for sid in test_ids})

        print("\n Salvataggio su disco...")
        count_files = 0
        for angle, angle_dict in self.data_store.items():
            for sid, seqs in angle_dict.items():
                split = id_to_split[sid]
                
                # Struttura: out_base / angle / split / file.npy
                save_dir = os.path.join(out_base, angle, split)
                os.makedirs(save_dir, exist_ok=True)
                
                for i, seq in enumerate(seqs):
                    # Nome file importante: ID_Split_Index.npy
                    fname = f"{sid}_{split}_{i}.npy"
                    np.save(os.path.join(save_dir, fname), seq)
                    count_files += 1

        print(f"\n Finito! Salvati {count_files} file .npy in {out_base}")
        print(f"   Train IDs: {len(train_ids)}")
        print(f"   Val IDs:   {len(val_ids)}")
        print(f"   Test IDs:  {len(test_ids)}")

#CONFIGURAZIONE DATASET
datasets = {
    "RealWorld": "/kaggle/input/realworld-gait",
    "LargeGait": "/kaggle/input/large-gait-dataset",
    "SMVDU": "/kaggle/input/smvdu-sg-dataset"
}

if __name__=="__main__":
    
    manager = GaitPreprocessor(target_frames=60)
    manager.run(datasets)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Scansione Dataset...
 Riepilogo iniziale:
 - Angle 90: 2187 video, 85 identità
 - Angle 0: 198 video, 22 identità
 - Angle 45: 973 video, 64 identità
 Elaborazione Video...


  0%|          | 0/3358 [00:00<?, ?it/s]


 Totale identità valide uniche: 91

 Salvataggio su disco...

 Finito! Salvati 3344 file .npy in /kaggle/working/processed_gait_multiangle
   Train IDs: 63
   Val IDs:   14
   Test IDs:  14


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
import os, glob, random
from collections import defaultdict

#CONFIGURAZIONE
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "/kaggle/input/processed-keypoints-yolo/processed_gait_multiangle"

TARGET_LEN = 60
EMBEDDING_DIM = 256
DROPOUT_RATE = 0.5 
EPOCHS = 180
LR_MODEL = 0.0005
LR_CENTER = 0.5    # Learning rate alto per i centri 
WEIGHT_DECAY = 0.0005 

# PK Sampler
P_PERSONS = 8
K_SAMPLES = 8

# Pesi Loss
LAMBDA_TRIPLET = 1.0
LAMBDA_CENTER = 0.005 # Il peso della Center Loss 

# CENTER LOSS MODULE
class CenterLoss(nn.Module):
    """Minimizza la distanza intra-classe mantenendo dei centroidi per ogni classe."""
    def __init__(self, num_classes, feat_dim, use_gpu=True):
        super(CenterLoss, self).__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.use_gpu = use_gpu
        
        
        if self.use_gpu:
            self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim).cuda())
        else:
            self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))

    def forward(self, x, labels):
        # x: feature matrix (batch_size, feat_dim)
        # labels: ground truth labels 
        batch_size = x.size(0)
        
        # Calcola distanza euclidea tra features e centri
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, self.num_classes) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(self.num_classes, batch_size).t()
        distmat.addmm_(x, self.centers.t(), beta=1, alpha=-2)

        # Seleziona solo le distanze relative alla classe corretta
        classes = torch.arange(self.num_classes).long()
        if self.use_gpu: classes = classes.cuda()
        labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
        mask = labels.eq(classes.expand(batch_size, self.num_classes))

        dist = distmat * mask.float()
        loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size
        return loss

# DATASET CON RANDOM MASKING
class GaitDataset(Dataset):
    def __init__(self, root_dir, split, augment=False):
        self.samples, self.augment = [], augment
        for angle in ["0", "45", "90"]:
            dirp = os.path.join(root_dir, angle, split)
            if not os.path.isdir(dirp): continue
            for f in glob.glob(os.path.join(dirp, "*.npy")):
                pid = os.path.basename(f).rsplit('_', 2)[0]
                self.samples.append((f, pid))
        self.pid_to_files = {}
        for f, pid in self.samples: self.pid_to_files.setdefault(pid, []).append(f)
        self.pids = sorted(list(self.pid_to_files.keys()))
        self.pid_to_idx = {pid: i for i, pid in enumerate(self.pids)}
        self.num_classes = len(self.pids)

    def _get_features(self, path):
        kp_all = np.load(path).astype(np.float32)
        T_orig = kp_all.shape[0]
        if T_orig != TARGET_LEN:
            new_idx = np.linspace(0, T_orig-1, TARGET_LEN)
            kp_new = np.zeros((TARGET_LEN, 17, 3), dtype=np.float32)
            for k in range(17):
                for c in range(3):
                    kp_new[:, k, c] = np.interp(new_idx, np.arange(T_orig), kp_all[:, k, c])
            kp_all = kp_new

        # AUGMENTATION
        if self.augment:
            # Noise e Mirror
            if random.random() < 0.5: kp_all[:, :, :2] += np.random.normal(0, 0.005, (TARGET_LEN, 17, 2))
            if random.random() < 0.5: kp_all[:, :, 0] *= -1 
            
            #Random Joint Masking (Simula occlusioni a vari angoli)
            # Maschera un intero arto (es. braccio sinistro) per tutto il video
            if random.random() < 0.3:
                # 5-6-7 (Braccio SX), 8-9-10 (Braccio DX), 11-13-15 (Gamba SX), 12-14-16 (Gamba DX)
                limbs = [[5,7,9], [6,8,10], [11,13,15], [12,14,16]]
                drop_limb = random.choice(limbs)
                kp_all[:, drop_limb, :] = 0.0

        hips = (kp_all[:, 11, :2] + kp_all[:, 12, :2]) / 2.0
        ankles = (kp_all[:, 15, :2] + kp_all[:, 16, :2]) / 2.0
        height = np.mean(np.linalg.norm(kp_all[:, 0, :2] - ankles, axis=1)) + 1e-6
        center_seq = np.mean(hips, axis=0)
        
        kp_xy = (kp_all[:, :, :2] - center_seq) / height
        vel = np.diff(kp_xy, axis=0, prepend=kp_xy[:1])
        features = np.concatenate([kp_xy, vel], axis=2).transpose(2, 0, 1) 
        return torch.from_numpy(features).float()

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, pid = self.samples[idx]
        return self._get_features(path), self.pid_to_idx[pid]

#MODELLO
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x): return F.avg_pool1d(x.clamp(min=self.eps).pow(self.p), (x.size(-1))).pow(1./self.p)

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.bn1 = nn.InstanceNorm2d(out_c, affine=True) 
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=(3, 1), padding=(1, 0), bias=False)
        self.bn2 = nn.InstanceNorm2d(out_c, affine=True)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=(3, 1), padding=(1, 0), bias=False)
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=1, bias=False), nn.InstanceNorm2d(out_c, affine=True))
    def forward(self, x): return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))

class GaitResNet(nn.Module):
    def __init__(self, in_channels=4, embedding_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(3, 3), padding=(1, 1), bias=False)
        self.bn1 = nn.InstanceNorm2d(64, affine=True)
        self.layer1 = BasicBlock(64, 64)
        self.layer2 = BasicBlock(64, 128)
        self.layer3 = BasicBlock(128, 256)
        self.gem = GeM()
        self.bn_head = nn.BatchNorm1d(256 * 17)
        self.dropout = nn.Dropout(DROPOUT_RATE)
        self.fc = nn.Linear(256 * 17, embedding_dim, bias=False)
        self.bn_out = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        B, C, T, V = x.shape
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C*V, T) 
        x = self.gem(x).view(B, -1)
        x = self.dropout(self.bn_head(x))
        return self.bn_out(self.fc(x))

# TRAINING LOOP DOPPIO
def batch_hard_triplet_loss_normalized(labels, embeddings, margin=0.3):
    embeddings = F.normalize(embeddings, p=2, dim=1)
    dist_mat = torch.cdist(embeddings, embeddings, p=2)
    mask_pos = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    hard_pos, _ = torch.max(dist_mat * mask_pos, dim=1)
    dist_mat_neg = dist_mat + (mask_pos * 1000.0)
    hard_neg, _ = torch.min(dist_mat_neg, dim=1)
    return F.relu(hard_pos - hard_neg + margin).mean()

# Sampler
class BalancedBatchSampler(Sampler):
    def __init__(self, dataset, p=8, k=8):
        self.dataset = dataset; self.p = p; self.k = k
        self.batch_size = p * k
        self.index_dic = defaultdict(list)
        for index, (_, pid_idx) in enumerate(dataset.samples): self.index_dic[pid_idx].append(index)
        self.pids = list(self.index_dic.keys())
        self.length = len(self.pids) // self.p * self.batch_size
    def __iter__(self):
        random.shuffle(self.pids)
        for i in range(0, len(self.pids) - self.p + 1, self.p):
            batch_indices = []
            for pid in self.pids[i : i + self.p]:
                idxs = self.index_dic[pid]
                if len(idxs) < self.k: batch_indices.extend(np.random.choice(idxs, self.k, replace=True))
                else: batch_indices.extend(np.random.choice(idxs, self.k, replace=False))
            yield batch_indices
    def __len__(self): return self.length // self.batch_size

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    feats, labels = [], []
    for x, lbl in loader:
        out = model(x.to(DEVICE))
        out = F.normalize(out, p=2, dim=1)
        feats.append(out.cpu()); labels.append(lbl)
    feats, labels = torch.cat(feats), torch.cat(labels)
    dist_m = 1.0 - torch.matmul(feats, feats.t())
    mask_same = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    intra = (dist_m * (mask_same - torch.eye(len(labels)))).sum() / (mask_same.sum() - len(labels) + 1e-6)
    inter = (dist_m * (1-mask_same)).sum() / ((1-mask_same).sum() + 1e-6)
    dist_m.fill_diagonal_(999)
    acc = (labels[dist_m.argmin(1)] == labels).float().mean().item() * 100
    return acc, intra.item(), inter.item()

def train():
    train_ds = GaitDataset(DATA_ROOT, "train", True)
    val_ds = GaitDataset(DATA_ROOT, "val", False)
    train_sampler = BalancedBatchSampler(train_ds, p=P_PERSONS, k=K_SAMPLES)
    train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, num_workers=2)

    model = GaitResNet(embedding_dim=EMBEDDING_DIM).to(DEVICE)
    
    # LOSSES
    triplet_loss = batch_hard_triplet_loss_normalized
    center_loss = CenterLoss(train_ds.num_classes, EMBEDDING_DIM, use_gpu=(DEVICE=="cuda"))

    # OPTIMIZERS 
    opt_model = optim.AdamW(model.parameters(), lr=LR_MODEL, weight_decay=WEIGHT_DECAY)
    opt_center = optim.SGD(center_loss.parameters(), lr=LR_CENTER) 

    sched_model = optim.lr_scheduler.MultiStepLR(opt_model, milestones=[60, 120, 160], gamma=0.1)

    print(f"🚀 Training CENTER-LOSS: Dim={EMBEDDING_DIM}")
    best_intra = 999 # Salva quando INTRA è bassa
    
    for epoch in range(EPOCHS):
        model.train()
        loss_trip_list, loss_cent_list = [], []
        
        for x, lbl in train_loader:
            x, lbl = x.to(DEVICE), lbl.to(DEVICE)
            
            # Forward
            embeds = model(x)
            
            # Calcolo Loss
            l_trip = triplet_loss(lbl, embeds)
            l_cent = center_loss(embeds, lbl)
            total_loss = (LAMBDA_TRIPLET * l_trip) + (LAMBDA_CENTER * l_cent)
            
            # Backward & Step
            opt_model.zero_grad()
            opt_center.zero_grad()
            total_loss.backward()
            
            # Clip gradient
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
              
            for param in center_loss.parameters():
                param.grad.data *= (1. / LAMBDA_CENTER) # Rescale gradiente centri
            
            opt_model.step()
            opt_center.step()
            
            loss_trip_list.append(l_trip.item())
            loss_cent_list.append(l_cent.item())
        
        sched_model.step()
        
        if (epoch+1) % 5 == 0:
            acc, intra, inter = evaluate(model, val_loader)
            gap = inter - intra
            print(f"Ep {epoch+1:03d} | Trip {np.mean(loss_trip_list):.3f} | Cent {np.mean(loss_cent_list):.3f} | Acc {acc:.2f}% | Gap {gap:.4f} (In:{intra:.3f}/Out:{inter:.3f})")
            
            # Salviamo se Intra-class distance migliora ed è decente l'accuracy
            if intra < best_intra and acc > 80.0:
                best_intra = intra
                torch.save(model.state_dict(), "gait_resnet_best.pth")
                print(f"⭐ BEST COMPACTNESS SAVED (Intra: {intra:.4f})")

if __name__ == "__main__":
    train()

🚀 Training CENTER-LOSS: Dim=256
Ep 005 | Trip 0.599 | Cent 324.336 | Acc 84.39% | Gap 0.1325 (In:0.607/Out:0.740)
⭐ BEST COMPACTNESS SAVED (Intra: 0.6072)
Ep 010 | Trip 0.549 | Cent 233.106 | Acc 84.39% | Gap 0.1863 (In:0.659/Out:0.846)
Ep 015 | Trip 0.557 | Cent 194.927 | Acc 88.07% | Gap 0.2442 (In:0.609/Out:0.853)
Ep 020 | Trip 0.547 | Cent 165.595 | Acc 86.14% | Gap 0.2849 (In:0.609/Out:0.894)
Ep 025 | Trip 0.564 | Cent 139.574 | Acc 84.74% | Gap 0.3037 (In:0.553/Out:0.857)
⭐ BEST COMPACTNESS SAVED (Intra: 0.5530)
Ep 030 | Trip 0.589 | Cent 126.186 | Acc 85.26% | Gap 0.2616 (In:0.409/Out:0.670)
⭐ BEST COMPACTNESS SAVED (Intra: 0.4088)
Ep 035 | Trip 0.565 | Cent 111.859 | Acc 85.61% | Gap 0.3287 (In:0.506/Out:0.835)
Ep 040 | Trip 0.593 | Cent 99.833 | Acc 82.98% | Gap 0.3439 (In:0.480/Out:0.824)
Ep 045 | Trip 0.591 | Cent 122.139 | Acc 87.19% | Gap 0.3341 (In:0.434/Out:0.768)
Ep 050 | Trip 0.567 | Cent 86.938 | Acc 84.39% | Gap 0.3210 (In:0.387/Out:0.708)
⭐ BEST COMPACTNESS SAVED (I